📌 1a. Load and normalize responses

In [ ]:
# 📥 Load the responses from the GRC questionnaire
import pandas as pd
from google.colab import files

# Replace this with your actual filename
df_resp = pd.read_csv("nist_controls_questions_answers.csv")

# Ensure optional columns exist before later steps
if "LLM Decision" not in df_resp.columns:
    df_resp["LLM Decision"] = pd.NA

# 👁️ Preview structure
print("📊 Columns:", df_resp.columns.tolist())
df_resp.head()


📌 1b. Normalize responses to three categories: Yes / No / Not Applicable

In [ ]:
# Normalize responses: strip, lowercase, and categorize
def normalize_response(resp):
    if not isinstance(resp, str):
        return "Unknown"
    resp = resp.strip().lower()
    if resp in ["yes", "y"]:
        return "Yes"
    elif resp in ["no", "n"]:
        return "No"
    elif "not applicable" in resp or resp.startswith("na"):
        return "Not Applicable"
    else:
        return "Unknown"

df_resp["Normalized Response"] = df_resp["Response"].apply(normalize_response)

# 🎯 Show distribution
df_resp["Normalized Response"].value_counts()

📌 2. Rule-based scoring (Yes = 1, No = 0, N/A = TBD)

In [ ]:
# 🎯 Assign initial binary score (Yes = 1, No = 0, Not Applicable = None)
def score_answer(resp):
    if resp == "Yes":
        return 1
    elif resp == "No":
        return 0
    else:
        return None  # "Not Applicable" or Unknown

df_resp["Binary Score"] = df_resp["Normalized Response"].apply(score_answer)

# 👁️ View how many are scored
df_resp["Binary Score"].value_counts(dropna=False)

📌 3a. LLM-assisted interpretation of “Not Applicable” + comment

In [ ]:
import openai

# Optional helper for interpreting Not Applicable responses.
# Keep this step separate from the core scoring flow because it depends on API access.
def interpret_na_with_llm(question, comment, model="gpt-3.5-turbo", temp=0):
    prompt = (
        "You are a cybersecurity auditor. Interpret the following Not Applicable justification to decide "
        "if the control is effectively implemented (YES), not implemented (NO), or truly Not Applicable (NA).\n\n"
        f"Question: {question}\n"
        f"User Justification: {comment}\n\n"
        "Reply with one word only: Yes, No, or NA."
    )
    try:
        response = openai.ChatCompletion.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are an expert in cybersecurity compliance."},
                {"role": "user", "content": prompt}
            ],
            temperature=temp,
            max_tokens=5,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {e}"


📌 3b. Apply the model to N/A answer

In [ ]:
# 📌 Filter in only N/A answer
df_na = df_resp[
    (df_resp["Normalized Response"] == "Not Applicable") &
    (df_resp["Comment"].notna()) &
    (df_resp["Comment"].str.strip() != "")
].copy()

print(f"✍️ Found {len(df_na)} rows to be interpreted with LLM")

# 🔁 Apply the model on the filtered answer
from tqdm import tqdm
tqdm.pandas()

df_na["LLM Decision"] = df_na.progress_apply(
    lambda row: interpret_na_with_llm(row["Question"], row["Comment"]),
    axis=1
)

📌 4. Compute Compliance Score from Evaluated Responses

In [ ]:
# Step 1 - Normalize Final Response Column

# Create a copy of the evaluated DataFrame
df_final = df_resp.copy()

# If LLM Decision is available and valid, overwrite the original response
mask_llm = (
    (df_final["Normalized Response"] == "Not Applicable")
    & (df_final["LLM Decision"].isin(["Yes", "No"]))
)
df_final.loc[mask_llm, "Normalized Response"] = df_final.loc[mask_llm, "LLM Decision"]

# Print final distribution of responses
df_final["Normalized Response"].value_counts()


In [ ]:
 #Step 2 – Compute Compliance Score per NIS2 Requirement

 # Group by each NIS2 requirement and count Yes / No answers
score_df = (
    df_final
    .groupby("Req ID")["Normalized Response"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

# Compute total questions and compliance score
score_df["Total Questions"] = score_df.get("Yes", 0) + score_df.get("No", 0)
score_df["Compliance Score"] = (score_df["Yes"] / score_df["Total Questions"]).round(2)

# Preview result
score_df[["Req ID", "Yes", "No", "Compliance Score"]].head()

In [ ]:
# Step 3 – Visualize Compliance Score Distribution

import matplotlib.pyplot as plt
import seaborn as sns

# Sort for better visualization
score_df_sorted = score_df.sort_values("Compliance Score", ascending=False)

# Bar plot: Compliance Score per NIS2 article
plt.figure(figsize=(10, 6))
sns.barplot(data=score_df_sorted, x="Compliance Score", y="Req ID", palette="viridis")
plt.title("Compliance Score per NIS2 Requirement", fontsize=14)
plt.xlabel("Compliance Score")
plt.ylabel("NIS2 Article")
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

📌 5. Classify Compliance Status per Requirement

In [ ]:
# Step 1 – Assign Compliance Level Based on Score

# Set custom thresholds
HIGH_THRESHOLD = 0.80 #to be decided
LOW_THRESHOLD = 0.50 #to be decided

# Assign final compliance status
def classify(score):
    if score >= HIGH_THRESHOLD:
        return "Compliant ✅"
    elif score >= LOW_THRESHOLD:
        return "Partially Compliant 🟡"
    else:
        return "Non-Compliant ❌"

score_df["Compliance Status"] = score_df["Compliance Score"].apply(classify)

# Show distribution
score_df["Compliance Status"].value_counts()

In [ ]:
#  Step 2 – Visualize Compliance Status Summary

plt.figure(figsize=(6, 4))
sns.countplot(data=score_df, x="Compliance Status", palette="Set2")
plt.title("Final Compliance Classification")
plt.xlabel("Compliance Status")
plt.ylabel("Number of Articles")
plt.grid(True, axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Step 3 – Export Classification Summary

# Save detailed compliance report
output_file = "nis2_compliance_results.csv"
score_df.to_csv(output_file, index=False)
print(f"✅ Results saved to '{output_file}'")
files.download(output_file)

VALUTARE REPORT DA PRODURRE